In [228]:
import torch
import torch.nn as nn
import torch.nn.functional
import os
import math
import time
import random
import warnings
import numpy as np
import numpy as np

In [229]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [230]:
from tokenizers import Tokenizer
from tqdm.auto import tqdm

In [231]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

In [232]:
from torch.cuda.amp import autocast
from torch.cuda.amp import GradScaler

In [233]:
warnings.filterwarnings("ignore")

In [234]:
SEED = 42

In [235]:
VOCAB_SIZE = 45_000
MAX_SEQ_LENGTH = 1024

In [236]:
MICRO_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 16
EPOCHS = 2

In [237]:
LEARNING_RATE = 5e-5
MIN_LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
GRAD_CLIP = 1.0
DROPOUT = 0.1

In [238]:
VALIDATE_EVERY = 500
SAVE_EVERY = 500
GENERATE_EVERY = 500
NUM_GENERATION_TOKENS = 256
USE_FP16 = True
NUM_WORKERS = 2
PIN_MEMORY = True
COMPILE_MODEL = False


In [239]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [240]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [241]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [242]:
BASE_MODEL_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-base-model/virgo_base_final.pt"

TRAIN_BIN = "/kaggle/input/datasets/punitkashyap2007/virgo-chat-final/train.bin"
VAL_BIN = "/kaggle/input/datasets/punitkashyap2007/virgo-chat-final/val.bin"

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

OUTPUT_DIR = "/kaggle/working/virgo_chat"

LAST_CHECKPOINT = os.path.join(OUTPUT_DIR, "virgo_chat_last.pt")
BEST_CHECKPOINT = os.path.join(OUTPUT_DIR, "virgo_chat_best.pt")
TRAINING_STATE = os.path.join(OUTPUT_DIR, "training_state.pt")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [243]:
device = torch.device(DEVICE)

In [244]:
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
else:
    print("Lmao poor you dont have anyyy GPU!!")
    gpu_name = 'cpu'
    total_memory = 0

In [245]:
scaler = GradScaler(enabled=USE_FP16)

In [246]:
print(f"Device                : {device}")
print(f"GPU                   : {gpu_name}")
print(f"GPU Memory            : {total_memory:.2f} GB")
print(f"Mixed Precision       : {USE_FP16}")
print(f"Micro Batch Size      : {MICRO_BATCH_SIZE}")
print(f"Gradient Accumulation : {GRADIENT_ACCUMULATION}")
print(f"Effective Batch Size  : {MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Max Sequence Length   : {MAX_SEQ_LENGTH}")

Device                : cuda
GPU                   : Tesla T4
GPU Memory            : 14.56 GB
Mixed Precision       : True
Micro Batch Size      : 8
Gradient Accumulation : 16
Effective Batch Size  : 128
Max Sequence Length   : 1024


In [247]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

In [248]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode='r')
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode='r')

In [249]:
train_tokens = len(train_data)
val_tokens = len(val_data)
total_tokens = train_tokens + val_tokens

In [250]:
tokens_per_step = MICRO_BATCH_SIZE * MAX_SEQ_LENGTH * GRADIENT_ACCUMULATION

steps_per_epoch = math.ceil(train_tokens / tokens_per_step)

total_steps = steps_per_epoch * EPOCHS

warmup_steps = int(total_steps * WARMUP_RATIO)

In [251]:
print(f"Vocabulary Size        : {VOCAB_SIZE:,}")
print(f"Train Tokens          : {train_tokens:,}")
print(f"Validation Tokens     : {val_tokens:,}")
print(f"Total Tokens          : {total_tokens:,}")

print()

print(f"Tokens / Optimizer Step : {tokens_per_step:,}")
print(f"Steps / Epoch          : {steps_per_epoch:,}")
print(f"Total Steps            : {total_steps:,}")
print(f"Warmup Steps           : {warmup_steps:,}")

Vocabulary Size        : 45,000
Train Tokens          : 889,659,392
Validation Tokens     : 8,868,864
Total Tokens          : 898,528,256

Tokens / Optimizer Step : 131,072
Steps / Epoch          : 6,788
Total Steps            : 13,576
Warmup Steps           : 407


In [252]:
class PackedDataset(torch.utils.data.Dataset):

    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return (len(self.data) - 1) // self.seq_length

    def __getitem__(self, idx):

        start = idx * self.seq_length
        end = start + self.seq_length + 1

        chunk = torch.from_numpy(
            np.array(self.data[start:end], dtype=np.int64)
        )

        x = chunk[:-1]
        y = chunk[1:]

        return x, y

In [253]:
train_dataset = PackedDataset(train_data, MAX_SEQ_LENGTH)
val_dataset = PackedDataset(val_data, MAX_SEQ_LENGTH)

In [254]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
    persistent_workers=NUM_WORKERS > 0
)

In [255]:
print(f"Training Samples   : {len(train_dataset):,}")
print(f"Validation Samples : {len(val_dataset):,}")

print()

print(f"Training Batches   : {len(train_loader):,}")
print(f"Validation Batches : {len(val_loader):,}")

Training Samples   : 868,807
Validation Samples : 8,660

Training Batches   : 108,600
Validation Batches : 1,083


In [256]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [257]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [258]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [259]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [260]:
class VirgoModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoModel, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [261]:
VOCAB_SIZE = 45000

D_MODEL = 768
NUM_HEADS = 12
NUM_LAYERS = 12
D_FF = 3072

MAX_SEQ_LENGTH = 1024
DROPOUT = 0.1

In [262]:
model = VirgoModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_seq_length=MAX_SEQ_LENGTH,
    dropout=DROPOUT
)

In [263]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"Total Parameters : {round(total_params / 1_000_000, 2)}M")
print(f"Trainable Parameters : {round(trainable_params / 1_000_000, 2)}M")


Total Parameters : 119.62M
Trainable Parameters : 119.62M


In [264]:
checkpoint = torch.load(BASE_MODEL_PATH, map_location="cpu")

state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint

In [265]:
missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print(f"Missing Keys         : {len(missing_keys)}")
print(f"Unexpected Keys      : {len(unexpected_keys)}")

assert len(missing_keys) == 0
assert len(unexpected_keys) == 0

Missing Keys         : 0
Unexpected Keys      : 0


In [266]:
model = model.to(device)

if COMPILE_MODEL:
    print("Compiling Model...")
    model = torch.compile(model)
    model = model.to(device)
print("\nVirgo Base Loaded Successfully!")


Virgo Base Loaded Successfully!


**Optimizer**

In [267]:
decay_params = []
no_decay_params = []

In [268]:
for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.ndim == 1 or name.endswith(".bias") or "norm" in name.lower():
        no_decay_params.append(param)
    else:
        decay_params.append(param)

In [269]:
optimizer = AdamW(
    [
        {
            "params": decay_params,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": no_decay_params,
            "weight_decay": 0.0
        }
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8
)

In [270]:
print(f"Learning Rate       : {LEARNING_RATE}")
print(f"Weight Decay        : {WEIGHT_DECAY}")
print(f"Decay Parameters    : {sum(p.numel() for p in decay_params):,}")
print(f"No Decay Parameters : {sum(p.numel() for p in no_decay_params):,}")

Learning Rate       : 5e-05
Weight Decay        : 0.01
Decay Parameters    : 119,494,656
No Decay Parameters : 121,344


In [271]:
def lr_lambda(current_step):

    if current_step < warmup_steps:
        return current_step / max(1, warmup_steps)

    progress = (current_step - warmup_steps) / max(
        1,
        total_steps - warmup_steps
    )

    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))

    min_lr_ratio = MIN_LEARNING_RATE / LEARNING_RATE

    return min_lr_ratio + (1 - min_lr_ratio) * cosine_decay

In [272]:
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

In [273]:
print(f"Initial LR     : {LEARNING_RATE:.2e}")
print(f"Minimum LR     : {MIN_LEARNING_RATE:.2e}")
print(f"Warmup Steps   : {warmup_steps:,}")
print(f"Total Steps    : {total_steps:,}")

Initial LR     : 5.00e-05
Minimum LR     : 5.00e-06
Warmup Steps   : 407
Total Steps    : 13,576


In [274]:
start_epoch = 0
global_step = 0
best_val_loss = float("inf")

In [275]:
def save_checkpoint(epoch, global_step, best_val_loss):

    # Save latest model weights
    torch.save(
        model.state_dict(),
        LAST_CHECKPOINT
    )

    # Save complete training state
    torch.save(
        {
            "epoch": epoch,
            "global_step": global_step,
            "best_val_loss": best_val_loss,

            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict()
        },
        TRAINING_STATE
    )

In [276]:
def load_checkpoint():

    global start_epoch
    global global_step
    global best_val_loss

    if not (
        os.path.exists(LAST_CHECKPOINT)
        and
        os.path.exists(TRAINING_STATE)
    ):
        print("No checkpoint found. Starting fresh.")
        return

    print("Loading checkpoint...")

    model.load_state_dict(
        torch.load(LAST_CHECKPOINT, map_location=device)
    )

    state = torch.load(TRAINING_STATE, map_location=device)

    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])

    start_epoch = state["epoch"]
    global_step = state["global_step"]
    best_val_loss = state["best_val_loss"]

    print(f"Resumed from Epoch {start_epoch + 1}")
    print(f"Global Step : {global_step:,}")
    print(f"Best Val Loss : {best_val_loss:.4f}")

In [277]:
load_checkpoint()

No checkpoint found. Starting fresh.


In [278]:
@torch.no_grad()
def evaluate(model, dataloader, max_batches=None):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    batches = 0

    for x, y in dataloader:

        if max_batches is not None and batches >= max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with autocast(enabled=USE_FP16):

            logits = model(x)

            loss = F.cross_entropy(
                logits.view(-1, VOCAB_SIZE),
                y.view(-1)
            )

        total_loss += loss.item()

        predictions = logits.argmax(dim=-1)

        total_correct += (predictions == y).sum().item()
        total_tokens += y.numel()

        batches += 1

    avg_loss = total_loss / batches
    accuracy = 100.0 * total_correct / total_tokens

    model.train()

    return avg_loss, accuracy

In [279]:
@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=NUM_GENERATION_TOKENS,
    temperature=0.8,
    top_k=50
):

    model.eval()

    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):

        input_cond = input_ids[:, -MAX_SEQ_LENGTH:]

        with autocast(enabled=USE_FP16):
            logits = model(input_cond)

        logits = logits[:, -1, :]
        logits = logits / temperature

        if top_k is not None:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == tokenizer.token_to_id("<eos>"):
            break

    model.train()

    return tokenizer.decode(input_ids[0].tolist())

In [280]:
SAMPLE_PROMPTS = [
    "Who are you?",
    "Explain machine learning.",
    "Write a Python function for quicksort.",
    "What is quantum mechanics?",
    "Solve: 2x + 5 = 17"
]

In [281]:
def generate_samples():

    print("\n" + "=" * 100)
    print("VIRGO CHAT SAMPLE")
    print("=" * 100)

    for prompt in SAMPLE_PROMPTS:

        print(f"\nUSER\n{'-' * 100}")
        print(prompt)

        print(f"\nVIRGO\n{'-' * 100}")

        response = generate(prompt)

        print(response)

        print("\n" + "=" * 100)

In [ ]:
print("=" * 100)
print("STARTING VIRGO CHAT TRAINING")
print("=" * 100)

tokens_seen = global_step * tokens_per_step
training_start = time.time()

for epoch in range(start_epoch, EPOCHS):

    model.train()

    epoch_loss = 0.0
    epoch_correct = 0
    epoch_tokens = 0

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        dynamic_ncols=True
    )

    for step, (x, y) in enumerate(pbar):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with autocast(enabled=USE_FP16):

            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )

            loss = loss / GRADIENT_ACCUMULATION

        scaler.scale(loss).backward()

        predictions = logits.argmax(dim=-1)

        batch_correct = (predictions == y).sum().item()
        batch_tokens = y.numel()

        current_acc = 100.0 * batch_correct / batch_tokens

        epoch_correct += batch_correct
        epoch_tokens += batch_tokens
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION

        if (step + 1) % GRADIENT_ACCUMULATION == 0:

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            global_step += 1
            tokens_seen += tokens_per_step

            current_lr = scheduler.get_last_lr()[0]

            train_loss = epoch_loss / (step + 1)
            train_acc = 100.0 * epoch_correct / epoch_tokens

            elapsed = time.time() - training_start
            tok_per_sec = tokens_seen / max(elapsed, 1e-6)

            if device.type == "cuda":
                vram = torch.cuda.memory_allocated() / 1024**3
            else:
                vram = 0

            remaining_steps = total_steps - global_step
            eta_seconds = remaining_steps * (elapsed / max(global_step, 1))
            eta_h = int(eta_seconds // 3600)
            eta_m = int((eta_seconds % 3600) // 60)

            pbar.set_postfix({
                "step": f"{global_step}/{total_steps}",
                "loss": f"{train_loss:.4f}",
                "curr": f"{current_acc:.2f}%",
                "cum": f"{train_acc:.2f}%",
                "lr": f"{current_lr:.2e}",
                "tok/s": f"{tok_per_sec/1000:.1f}K",
                "VRAM": f"{vram:.2f}GB",
                "ETA": f"{eta_h}h {eta_m}m"
            })

            if global_step % VALIDATE_EVERY == 0:

                val_loss, val_acc = evaluate(
                    model,
                    val_loader,
                    max_batches=100
                )

                print("\\n" + "=" * 100)
                print(f"STEP {global_step:,}")
                print("=" * 100)
                print(f"Train Loss       : {train_loss:.4f}")
                print(f"Current Accuracy : {current_acc:.2f}%")
                print(f"Cumulative Acc   : {train_acc:.2f}%")
                print()
                print(f"Validation Loss  : {val_loss:.4f}")
                print(f"Validation Acc   : {val_acc:.2f}%")
                print()
                print(f"Learning Rate    : {current_lr:.2e}")
                print(f"Optimizer Step   : {global_step:,}/{total_steps:,}")
                print(f"Tokens Seen      : {tokens_seen:,}")
                print(f"Tokens / Second  : {tok_per_sec:,.0f}")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), BEST_CHECKPOINT)
                    print("\\n✓ Best model updated.")

            if global_step % GENERATE_EVERY == 0:
                generate_samples()

            if global_step % SAVE_EVERY == 0:
                save_checkpoint(epoch, global_step, best_val_loss)

    print("\\n" + "=" * 100)
    print(f"END OF EPOCH {epoch+1}")
    print("=" * 100)

    full_val_loss, full_val_acc = evaluate(model, val_loader)

    print(f"Validation Loss : {full_val_loss:.4f}")
    print(f"Validation Acc  : {full_val_acc:.2f}%")

    if full_val_loss < best_val_loss:
        best_val_loss = full_val_loss
        torch.save(model.state_dict(), BEST_CHECKPOINT)
        print("✓ Best model updated.")

    save_checkpoint(epoch + 1, global_step, best_val_loss)

print("\\nTraining Complete!")

STARTING VIRGO CHAT TRAINING


Epoch 1/2:   0%|          | 0/108600 [00:00<?, ?it/s]